In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks, peak_widths, savgol_filter
import pandas as pd
import h5py
from typing import Literal
import struct

plt.rcParams["figure.dpi"] = 256

In [ ]:
def plot_with_peaks(
    data,
    n_bins,
    prominence=None,
    xlabel="",
    title="",
    xlims=None,
    savgol_window=None,
    savgol_strength=1.0,
    clip_counts=None,
    stacked=False,
    legend=None,
    bin_range=None,
    text_size=5,
):
    if not stacked:
        signals = [data]
    else:
        signals = data

    bottom = np.zeros(n_bins)

    for signal in signals:
        counts, bins = np.histogram(signal, bins=n_bins, density=False, range=bin_range)
        if savgol_window is not None:
            background = savgol_filter(counts, window_length=savgol_window, polyorder=3)
            counts = counts - background * savgol_strength

        if clip_counts is not None:
            counts[counts > clip_counts] = clip_counts

        bin_width = np.diff(bins).mean()
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        height_scale = counts.max()

        plt.bar(bin_centers, counts, width=bin_width, align="center", bottom=bottom)
        plt.xlim(xlims)
        bottom += counts
        bottom = np.round(bottom, 10)
        if prominence is not None:
            peaks, props = find_peaks(counts, prominence=prominence)
            half_widths = peak_widths(counts, peaks)[0] * bin_width
            for x, y, half_width in zip(bin_centers[peaks], counts[peaks], half_widths):
                plt.errorbar(
                    [x],
                    [y],
                    xerr=[half_width],
                    c="black",
                    fmt="",
                    capsize=2,
                    elinewidth=0.2,
                    capthick=0.2,
                )
                plt.annotate(
                    f"{x:.03f} ± {half_width:.03f} MeV",
                    (x, y + height_scale * (0.005 * text_size)),
                    ha="left",
                    fontsize=text_size,
                )

    plt.xlabel(xlabel)
    plt.ylabel("Count")
    # plt.ylim(ylims)
    plt.title(title)
    if legend is not None:
        plt.legend(legend)

    # plt.figure()
    plt.show()

# Loaders

In [ ]:
keys = [
    "E",
    "name",
    "z",
    "rho",
    "phi",
    "momentumZ",
    "momentumRho",
    "momentumPhi",
    "sourceDepth",
    "sourceRadius",
    "sourceE",
    "sourcePDG"
]

def decode(b):
    return b.decode("utf-8")


decode_v = np.vectorize(decode)


def load_hdf5(path):
    dfs = []
    with h5py.File(path, "r") as f:
        for key in keys:
            data = f["default_ntuples"]["Measurement"][key]["pages"][()]
            if isinstance(data[0], bytes):
                data = decode_v(data)

            dfs.append(pd.DataFrame(data, columns=[key]))

    return pd.concat(dfs, axis=1)


def load_csv(path):
    df = pd.read_csv(path, comment="#", header=None)
    df.columns = keys.copy()
    print(f"{df.shape[0]:,} interactions recorded")
    return df


def load_raw(path):
    rows = []
    with open(path, "rb") as file:
        while data := file.read(3 * 8):
            energy, p_z, depth = struct.unpack("ddd", data)
            rows.append([energy, p_z, depth])
    df = pd.DataFrame(rows, columns=["E", "p_z", "z"])
    return df

In [ ]:
df = load_csv("finals/big.csv")
df

# Analysis

In [ ]:
particle_names, particle_counts = np.unique(df["name"], return_counts=True)
# particle_nums = np.arange(0, len(particle_names))
# particle_dict = dict(zip(particle_names,particle_nums))

particle_counts, particle_names = zip(
    *sorted(zip(particle_counts, particle_names))[::-1]
)

names_dict = {
    "gamma": "$\\gamma$",
    "e-": "$e^{-}$",
    "e+": "$e^{+}$",
    "neutron": "neutron",
    "proton": "proton",
    "nu_e": "$\\nu_e$",
    "anti_nu_e": "Anti-$\\nu_e$",
    "nu_mu": "$\\nu_\\mu$",
    "anti_nu_mu": "Anti-$\\nu_\\mu$",
    "pi+": "$\\pi^{+}$",
}


In [ ]:
plt.yscale("log")

plt.bar([names_dict[name] for name in particle_names], particle_counts)
plt.xticks(rotation=45, ha='right')
plt.title("Particles Incident on Detector Surface")
plt.ylabel("Count")

In [ ]:
gamma_prop = particle_counts[0] / np.sum(particle_counts)
gamma_prop_u = np.sqrt(particle_counts[0]) / np.sum(particle_counts)
print(f"{gamma_prop:.04f} \\pm {gamma_prop_u:.04f}")

e_prop = particle_counts[1] / np.sum(particle_counts)
e_prop_u = np.sqrt(particle_counts[1]) / np.sum(particle_counts)
print(f"{e_prop:.05f} \\pm {e_prop_u:.05f}")

## Masking Utilities

In [ ]:
property_range = lambda property, lower, upper, mask=None: (
    df if mask is None else df[mask]
)[property].between(lower, upper)
energy_range = lambda lower, upper, mask=None: property_range("E", lower, upper, mask)

In [ ]:
# plot_with_peaks(df["sourceDepth"], 100)
# plot_with_peaks(df["sourceRadius"], 100)
# plt.yscale("log")
# plot_with_peaks(df["z"], 100)
# plot_with_peaks(df["rho"], 100)
# plot_with_peaks(df["phi"], 100)
# plot_with_peaks(df["momentumZ"], 100)
# plot_with_peaks(df["momentumRho"], 100)
mask = df["z"].between(200, 368)
diff = (df["phi"]-df["momentumPhi"]) % (2 * np.pi) - np.pi
plot_with_peaks(diff, 2048, xlabel="Momentum Angle from Radial")
df[mask]

In [ ]:
plt.rcParams["figure.dpi"] = 500
plot_with_peaks(df[energy_range(2.0, 10.0)]["E"], n_bins=2048, clip_counts=4000)

In [ ]:
plt.close()
mask = df["z"].between(0, 360)
plt.hexbin(
    df[mask]["rho"] / 10 * np.cos(df[mask]["phi"]),
    df[mask]["rho"] / 10 * np.sin(df[mask]["phi"]),
    # C=df["E"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    # norm=LogNorm(),
)
plt.colorbar(label="Count")
plt.xlabel("$x$ (cm)")
plt.ylabel("$y$ (cm)")
plt.title("Positions of Particles Resulting in Detector Pulses")
plt.show()

plt.hexbin(
    df[mask]["sourceRadius"] / 10,
    df[mask]["sourceDepth"] / 10,
    # C=df["E"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    # norm=LogNorm(),
)
plt.colorbar(label="Count")
plt.xlabel("Distance from $z$-axis, $\\rho$ (cm)")
plt.ylabel("$z$ (cm)")
plt.title("Positions of Particles Resulting in Detector Pulses")
plt.show()

In [ ]:
num_in_vol = df[(df["sourceDepth"] < 500) & (df["sourceRadius"] < 2500)].size
num_in_vol_u = np.sqrt(num_in_vol)
num_tot = df.size
prop = num_in_vol/num_tot
prop_u = num_in_vol_u/num_tot

print(f"{prop:.01%} +/- {prop_u:.01%}")


In [ ]:
plot_with_peaks(df["sourceRadius"], 128)

In [ ]:
df["sourceDepth"].mean()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

mask = ((df["momentumPhi"] - df["phi"]) % (2 * np.pi)).abs() < np.pi/2
d = df

# ax.quiver(
#     d["rho"] * np.cos(d["phi"]),
#     d["rho"] * np.sin(d["phi"]),
#     d["z"],
#     d["momentumRho"] * np.cos(d["momentumPhi"]),
#     d["momentumRho"] * np.sin(d["momentumPhi"]),
#     d["momentumZ"],
#     length=50,
#     linewidth=0.1,
#     arrow_length_ratio=0.3
# )
scatter_plot = ax.scatter(
    d["rho"] / 10 * np.cos(d["phi"]),
    d["rho"] / 10 * np.sin(d["phi"]),
    d["z"] / 10,
    s=0.01,
    c=d["z"],#d["sourceDepth"],#(d["phi"]-d["momentumPhi"]) % (2.0 * np.pi) - np.pi,
    depthshade=False,
    alpha=0.8,
    linewidths=0.1
)

ax.set_title("Particles Recorded in Pre-Detector Transport Stage")

cb = fig.colorbar(scatter_plot, ax=ax)
cb.set_label("$z$ (cm)")

ax.set_xlabel('$x$ (cm)')
ax.set_ylabel('$y$ (cm)')
ax.set_zlabel('$z$ (cm)')

plt.show()